# 03 · MIDI를 학습 이미지로 변환
원본: Music_gen/Creating_data_set_from_midifiles.ipynb

단일 트랙 MIDI를 8마디의 128×128 피아노롤로 만든 뒤, 음역을 64행으로 줄이고 4마디씩 나눕니다. 곡 하나에서 64×64 이미지 두 개를 만듭니다. 원본 함수는 보존하고 디버그 셀을 제거했습니다.

In [ ]:
%pip install mido
from mido import MidiFile
import numpy as np
from pathlib import Path

## 입력 경로
Colab 파일 패널에서 `/content/project/data/midi`에 직접 만든 MIDI를 넣으세요. Drive를 쓰면 아래 BASE를 마운트한 프로젝트 폴더로 바꾸세요. 파일은 하위 폴더까지 찾습니다.

In [ ]:
BASE = Path('/content/project')
MIDI_DIR = BASE / 'data' / 'midi'
OUTPUT = BASE / 'data' / 'data_set_64.npy'
MIDI_DIR.mkdir(parents=True, exist_ok=True)
pathes_list = sorted(str(p) for p in MIDI_DIR.rglob('*.mid'))
if not pathes_list:
    raise FileNotFoundError(f'MIDI 파일을 넣어 주세요: {MIDI_DIR}')

In [ ]:
def read_midi_file(midi_file_path, ticks_per_beat=480):
    # print('midi_file_path', midi_file_path)
    mid = MidiFile(midi_file_path, ticks_per_beat=ticks_per_beat)
    note_temp = []
    note_import = []
    tick = 0
    for i, track in enumerate(mid.tracks):
        for msg in track:
            tick += msg.time
            if msg.type == 'note_on' or msg.type == 'note_off':
                if msg.type == 'note_on'and msg.velocity !=0:
                    pitch = msg.note
                    start = tick*4 / float(mid.ticks_per_beat)
                    start = float("{:.4f}".format(start))  # 시간 퀀타이즈
                    velocity = msg.velocity
                    note_info = {
                        'pitch': pitch,
                        'start_time': start,
                        'length': None,
                        'velocity': velocity,
#                         'function': None,
                    }
                    note_temp.append(note_info)
                elif msg.type == 'note_off'or msg.velocity == 0:
                    for j in range(len(note_temp)):
                        if note_temp[j]['pitch'] == msg.note:
                            end = tick*4 / float(mid.ticks_per_beat)
                            note_temp[j]['length'] = end - note_temp[j]['start_time']
                            note_import.append(note_temp[j])
                            del note_temp[j]
                            break
            else:
                pass
                # print(msg)

    note_import = sorted(note_import, key=lambda k: k['start_time'])
    return note_import

In [ ]:
#제거할 피치 리스트
remove_these_rows = []
for i in range(36):
  remove_these_rows.append(i)
for i in range(60,72):
  remove_these_rows.append(i)
for i in range(112, 128):
  remove_these_rows.append(i)

remove_these_rows

In [ ]:
def make_data_set_64(pathes_list):
  data_set = []

  for path in pathes_list:
    song = read_midi_file(path)
    temp_data = np.zeros((128,128))
    if song[-1]['start_time'] >= 128: #9마디 짜리이면
      for note in song:
        if not note['start_time'] < 16.0: #못 갖춘 마디 버림
          x = int(note['pitch'])
          y = int(round(note['start_time']))
          length = int(round(note['length']))
          for i in range(length):
            if y+i-16 > 127:
              pass
            else:
              temp_data[x][y+i-16] = 1
    else: #8마디 짜리이면
      for note in song:
        x = int(note['pitch'])
        y = int(round(note['start_time']))
        length = int(round(note['length']))
        for i in range(length):
          if y+i > 127:
            pass
          else:
            temp_data[x][y+i] = 1
    temp_data = np.delete(temp_data, remove_these_rows, axis=0)
    temp_data1 = np.delete(temp_data, np.s_[64::1], axis=1)
    temp_data2 = np.delete(temp_data, np.s_[:64:1], axis=1)
    data_set.append(temp_data1)
    data_set.append(temp_data2)

  data_set = np.array(data_set)
  
  return data_set

In [ ]:
data_set = make_data_set_64(pathes_list)
assert data_set.shape == (2 * len(pathes_list), 64, 64)
assert np.isin(data_set, [0, 1]).all()
np.save(OUTPUT, data_set)
print('입력 MIDI:', len(pathes_list), '학습 이미지:', data_set.shape, '저장:', OUTPUT)